In [1]:
# ---------------------------------------------------------
# 1. PDF Source Configuration
# ---------------------------------------------------------

import os, sys, time, warnings, logging
import fitz, cv2
import numpy as np

PDF_REL_PATH = "data/raw/vie/an-nam-chi-nguyen/HVB_002_PDFScan_Viet_An Nam Chí Nguyên.pdf"
WORK_ID      = "HVB_002"
WORK_TITLE   = "An Nam Chí Nguyên"

START_PAGE   = 10   # INDEX
NUM_PAGES    = 1   # OFFSET (None for all)
USE_LLM      = True # LLM for corrector
# ---------------------------------------------------------

# Tự động kết nối đường dẫn tới các helper trong src/run_ocr/vie
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
src_ocr_path = os.path.abspath(os.path.join(current_dir, "..", "src", "run_ocr", "vie"))
if src_ocr_path not in sys.path:
    sys.path.insert(0, src_ocr_path)

from ocr_utils import (
    find_file, enhance_image, init_paddleocr, init_vietocr,
    run_ocr_page, smart_sort_layout, clean_viet_text
)
from llm_corrector import correct_text_with_llm

print("✅ Đã import thành công & sẵn sàng cấu hình pipeline!")

✅ Đã import thành công & sẵn sàng cấu hình pipeline!


In [2]:
# ---------------------------------------------------------
# 2. LOAD & RENDER PDF TO IMAGES
# ---------------------------------------------------------

pdf_path = os.path.abspath(os.path.join(current_dir, "..", PDF_REL_PATH))
if not os.path.exists(pdf_path):
    pdf_path = find_file(PDF_REL_PATH, WORK_ID)

if not pdf_path or not os.path.exists(pdf_path):
    raise FileNotFoundError(f"❌ Không tìm thấy file PDF tại: {PDF_REL_PATH}")

print(f"📂 Đang mở file PDF: {os.path.basename(pdf_path)}")
doc = fitz.open(pdf_path)
total_pages = len(doc)

end_page = min(START_PAGE + NUM_PAGES, total_pages) if NUM_PAGES else total_pages
print(f"📑 Sẽ chạy từ trang {START_PAGE + 1} đến trang {end_page} (Tổng: {end_page - START_PAGE} trang)...")

pages_images = []
for i in range(START_PAGE, end_page):
    page = doc[i]
    mat = fitz.Matrix(2.5, 2.5)  # Scale 2.5x ~ 180-250 DPI
    pix = page.get_pixmap(matrix=mat)
    
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)
    pages_images.append(img)

doc.close()
print(f"✅ Đã render xong {len(pages_images)} trang ảnh!")

📂 Đang mở file PDF: HVB_002_PDFScan_Viet_An Nam Chí Nguyên.pdf
📑 Sẽ chạy từ trang 11 đến trang 11 (Tổng: 1 trang)...
✅ Đã render xong 1 trang ảnh!


In [3]:
# ---------------------------------------------------------
# 3. KHỞI TẠO ENGINE OCR (PADDLE + VIETOCR)
# ---------------------------------------------------------

print("⏳ Khởi tạo PaddleOCR (Detector)...", end=" ", flush=True)
paddle_engine = init_paddleocr(lang="vi")
print("OK")

print("⏳ Khởi tạo VietOCR (vgg_seq2seq Recognizer)...", end=" ", flush=True)
vietocr_engine = init_vietocr()
print("OK ✅")


⏳ Khởi tạo PaddleOCR (Detector)... 

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\James\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\James\.paddlex\official_models\UVDoc`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\James\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\James\.paddlex\official_models\PP-OCRv6_medium_rec`.


OK
⏳ Khởi tạo VietOCR (vgg_seq2seq Recognizer)...   → VietOCR weights: [local] vgg_seq2seq.pth
OK ✅


In [4]:
# ---------------------------------------------------------
# 4. PIPELINE OCR + LLM CORRECTION
# ---------------------------------------------------------

result_pages = []
total_t0 = time.time()

print(f"\n🚀 Bắt đầu xử lý {len(pages_images)} trang...")
for idx, img in enumerate(pages_images):
    t0 = time.time()
    print(f"  → [Trang {idx + 1}/{len(pages_images)}] Đang nhận diện...", end=" ", flush=True)
    
    # 1. Tiền xử lý ảnh (CLAHE + Unsharp Masking)
    enhanced = enhance_image(img)
    
    # 2. Paddle detect box → crop → VietOCR đọc text
    ocr_lines = run_ocr_page(enhanced, paddle_engine, vietocr_engine)
    
    # 3. Sắp xếp layout chuẩn từ trên xuống dưới, trái qua phải
    page_text = smart_sort_layout(ocr_lines)
    
    # 4. Hậu xử lý LLM sửa lỗi OCR (nếu USE_LLM = True)
    if USE_LLM:
        try:
            page_text = correct_text_with_llm(page_text, work_title=WORK_TITLE, language="vie")
        except Exception as e:
            print(f"(LLM lỗi: {e}, dùng text Raw OCR)", end=" ")
    
    print(f"Xong ({time.time() - t0:.1f}s) - {len(ocr_lines)} dòng.")

elapsed_total = time.time() - total_t0
print(f"\n🎉 HOÀN TẤT PIPELINE! Tổng thời gian: {elapsed_total:.2f} giây (~{elapsed_total/len(pages_images):.1f}s/trang).")



🚀 Bắt đầu xử lý 1 trang...
  → [Trang 1/1] Đang nhận diện...   LLM chunk 1/1... OK
Xong (74.2s) - 32 dòng.

🎉 HOÀN TẤT PIPELINE! Tổng thời gian: 74.16 giây (~74.2s/trang).


In [7]:
print("\n--- All Lines ---")
for idx, line in enumerate(ocr_lines):
    box, (text, score) = line
    print(f"{idx+1:2d}. [{score*100:5.1f}%] : {text}")


--- All Lines ---
 1. [100.0%] : CONTERSION
 2. [100.0%] : THẾ LỆ HIỆU CHU
 3. [100.0%] : 0010000001010
 4. [100.0%] : Bản dịch An Nam chỉ nguyên do có học giả, dịch giả Hoa Bằng
 5. [100.0%] : Hoàng Thúc Trâm hoàn thành từ cách đây hơn nửa thế kỉ (1961)
 6. [100.0%] : Trước đó tập sách này mới chỉ được trích dịch và công bố
 7. [100.0%] : một phần nhưng từ khi được Hoa Bằng dịch trọn ven (1961)
 8. [100.0%] : đếm nay, bản thảo chí tốn tai dưới dang in rành nội bộ
 9. [100.0%] : Nguyên và viện và đường như rất được biết đến được biết đến được biết được biết đến được biết được biết đến được biết được biết đ
10. [100.0%] : nguyên là không quận trong quốc trong tháng tháng tháng bản tháo đến tháng được nhanh tháng được nhanh tháng được nhanh thánh thá
11. [100.0%] : Thi thi thi thi thi tri chi một nha một nha một như thanh được như chiến được như chiến được như chiến được như chiến được như ch
12. [100.0%] : Sai sót (về chính tà, dấu cầu, thanh điệu, nội dung
13. [100.0%] : đứt đoạn, thi

In [5]:
print(page_text)

THỂ LỆ HIỆU CHÚ
Bản dịch An Nam chí nguyên do cố học giả, dịch giả Hoa Bằng
Hoàng Thúc Trâm hoàn thành từ cách đây hơn nửa thế kỉ (1961).
Trước đó tập sách này mới chỉ được trích dịch và công bố
một phần nhưng từ khi được Hoa Bằng dịch trọn vẹn (1961)
đến nay, bản thảo chỉ tồn tại dưới dạng in rônêô nội bộ
của Viện Sử học và dường như rất ít được biết đến.
Nguyên bản thảo do thời gian lưu trữ đã lâu nên tình trạng vật lý
bị xuống cấp, chữ mờ, giấy mục, nhiều trang bị rách. Hơn


In [ ]:
# # ---------------------------------------------------------
# # 4. EXPORT TEXT TO .TXT
# # ---------------------------------------------------------

# output_dir = os.path.abspath(os.path.join(current_dir, "..", "data", "ocr_output"))
# os.makedirs(output_dir, exist_ok=True)

# out_file = os.path.join(output_dir, f"{WORK_ID}_test_vie_raw.txt")
# final_full_text = clean_viet_text("\n\n".join(result_pages))

# with open(out_file, "w", encoding="utf-8") as f:
#     f.write(final_full_text)

# print(f"💾 Đã lưu kết quả hoàn chỉnh vào:\n  👉 {out_file}\n")
# print("-" * 50)
# print("📋 PREVIEW 1000 KÝ TỰ ĐẦU TIÊN:")
# print("-" * 50)
# print(final_full_text[:1000])


💾 Đã lưu kết quả hoàn chỉnh vào:
  👉 d:\HCMUS\HP2\MTH020-NLP\nlp-k35-midterm\src\run_ocr\data\ocr_output\HVB_002_test_vie_raw.txt

--------------------------------------------------
📋 PREVIEW 1000 KÝ TỰ ĐẦU TIÊN:
--------------------------------------------------

